# LAB- P8: El Modelo Neoclásico de Crecimiento Exógeno (Solow-Swan) (Julia)
**Proyecto MACRO-AI-COMP (Convocatoria INNOVA26, UMA / Banco Santander)**
*   **Código de Práctica**: LAB-P8-v1.0
*   **Capítulo de Referencia**: Capítulo 9, *An Introduction to Computational Macroeconomics* (Bongers, Gómez y Torres, Vernon Press, 2019)
*   **Autores**: Equipo Docente MACRO-AI-COMP
*   **Objetivo**: Simular y analizar la dinámica de acumulación de capital en tiempo discreto, el proceso de transición hacia el estado estacionario, los efectos de perturbaciones estructurales (tasa de ahorro, demografía y TFP), y el principio de la Regla de Oro.

---

## Objetivos de Aprendizaje
Al finalizar esta práctica, serás capaz de:
1.  **Comprender** el mecanismo dinámico de acumulación de capital físico per cápita propuesto por Solow (1956) y Swan (1956).
2.  **Identificar** los determinantes de largo plazo de la riqueza y productividad laboral en el estado estacionario (tasa de ahorro, depreciación, crecimiento poblacional y TFP).
3.  **Analizar** la transición dinámica de las variables per cápita (capital, producción, consumo e inversión) tras shocks estructurales.
4.  **Diferenciar** entre el efecto impacto (corto plazo) y el efecto de largo plazo sobre el bienestar tras un incremento del ahorro.
5.  **Explicar** analítica y visualmente el concepto de la **Regla de Oro** y las consecuencias de la ineficiencia dinámica (infra-acumulación vs. sobre-acumulación).


> **👋 BIENVENIDA A LA PRÁCTICA - LEER ANTES DE EMPEZAR**
> 
> *   **¿Nunca has usado Jupyter?** No te preocupes. Este cuaderno es interactivo. Haz clic en cualquier celda de código y pulsa **`Shift + Enter`** para ejecutarla. Ve de arriba a abajo en orden.
> *   **¿Se ha congelado o sale un asterisco `[*]` eterno?** Ve al menú superior y dale a `Kernel` ➔ `Restart`.
> *   **El objetivo** de esta práctica es que juegues con la economía. Cambia los números del código que representan impuestos, dinero o tecnología, vuelve a ejecutar y mira los gráficos. ¡No puedes romper nada!
>

### 🕹️ GUÍA RÁPIDA PARA DUMMIES - Modelo de Crecimiento de Solow-Swan
*   **¿Qué estamos haciendo aquí?** Analizando por qué crecen los países a largo plazo y cómo acumulan capital (maquinaria).
*   **La Regla de Oro:** Ahorrar más es bueno porque permite comprar más máquinas, pero si ahorras demasiado, no te queda dinero para consumir. Existe una "tasa de ahorro óptima" que maximiza el bienestar.
*   **¡Prueba esto!** Sube la tasa de ahorro y observa cómo la economía transiciona hacia un nivel de riqueza permanente más alto.


In [ ]:
# Este cuaderno depende del paquete `MacroAIComp` (Project.toml/Manifest.toml
# en la raíz del repositorio). En MyBinder (ver docs/DESPLIEGUE_BINDER.md) y en
# tu entorno local, el kernel ya arranca dentro del repositorio clonado, así
# que la celda siguiente activa e instancia el proyecto automáticamente.
# Nota: Google Colab no soporta Julia de forma nativa desde un notebook .ipynb;
# para la versión Julia de esta práctica usa MyBinder.


In [ ]:
using Pkg
Pkg.activate("../..")
Pkg.instantiate()

using MacroAIComp
using Plots
import Plots: mm
default(gridalpha=0.6, gridstyle=:dot)  # estilo de grid consistente con la versión Python
using LinearAlgebra
using Interact
using BenchmarkTools


## 1. El Modelo Neoclásico de Crecimiento (Solow-Swan)

El modelo de Solow-Swan es el bloque fundamental de la teoría moderna del crecimiento económico. A diferencia de los modelos de equilibrio general dinámico con agentes optimizadores (como Ramsey o DGE básico), este modelo asume que **la tasa de ahorro es exógena y constante**. 

### 1.1 Estructura Matemática

El modelo se define en variables per cápita (por trabajador) para aislar el tamaño de la economía y capturar el bienestar individual:
*   **Población (Trabajo):** Crece a una tasa constante $n$:
    $$L_{t} = L_{t-1}(1 + n)$$
*   **Función de Producción Intensiva (Cobb-Douglas):**
    $$y_t = A_t k_t^\alpha$$
    donde $y_t \equiv Y_t/L_t$ es la producción per cápita, $k_t \equiv K_t/L_t$ es el stock de capital per cápita, $A_t$ es la TFP, y $\alpha$ es la elasticidad capital-trabajo.
*   **Ahorro e Inversión:** Una fracción constante $s$ de la producción se destina a la inversión bruta:
    $$i_t = s y_t$$
*   **Consumo per cápita:** Lo que no se ahorra se consume:
    $$c_t = (1 - s)y_t$$

### 1.2 Dinámica del Capital y Estado Estacionario
La acumulación de capital per cápita sigue la siguiente ecuación en diferencias de primer orden:
$$k_{t+1} = \frac{(1 - \delta)k_t + s A_t k_t^\alpha}{1 + n}$$

El estado estacionario ($\bar{k}$) ocurre cuando el capital por trabajador permanece constante ($\Delta k_t = 0$), lo que implica que la inversión por trabajador cubre exactamente la depreciación física y el crecimiento de la población (depreciación efectiva):
$$s A \bar{k}^\alpha = (\delta + n)\bar{k}$$

Despejando el capital per cápita estacionario:
$$\bar{k} = \left( \frac{\delta + n}{s A} \right)^{\frac{1}{\alpha - 1}}$$

Una vez obtenido $\bar{k}$, las demás variables se calculan de manera directa:
$$\bar{y} = A \bar{k}^\alpha, \quad \bar{c} = (1 - s)\bar{y}, \quad \bar{i} = s\bar{y}$$


In [ ]:
params_base = default_calibration(SolowSwanParameters)
ss = compute_solow_steady_state(params_base)

println("VALORES DE ESTADO ESTACIONARIO:")
println("  Capital por trabajador (k*)   : ", round(ss["k"], digits=4))
println("  Producción por trabajador (y*): ", round(ss["y"], digits=4))
println("  Consumo por trabajador (c*)   : ", round(ss["c"], digits=4))
println("  Inversión por trabajador (i*) : ", round(ss["i"], digits=4))


## 2. Simulación Interactiva: Transición Dinámica y Shocks

Supongamos que la economía parte de su estado estacionario base con una tasa de ahorro del $20\%$ ($s_0 = 0.20$), depreciación del $6\%$ ($\delta = 0.06$) y crecimiento poblacional del $2\%$ ($n_0 = 0.02$). En el período $t = 5$, la economía sufre una perturbación permanente en su tasa de ahorro (por ejemplo, sube al $25\%$).

Con el siguiente panel interactivo, puedes mover la tasa de ahorro final ($s_1$), el crecimiento poblacional final ($n_1$), y la TFP final ($A_1$) para analizar cómo responde la economía período a período.


In [ ]:
# Simulación interactiva: Transición Dinámica de Solow
@manipulate for s_final in 0.10:0.01:0.50, n_final in 0.00:0.005:0.05, A_final in 0.5:0.1:2.0
    
    params_init = default_calibration(SolowSwanParameters)
    ss_init = compute_solow_steady_state(params_init)
    k0 = ss_init["k"]
    T_sim = 100
    
    # Send shocks at t=5
    s_path = fill(params_init.s, T_sim)
    s_path[6:end] .= s_final
    
    n_path = fill(params_init.n, T_sim)
    n_path[6:end] .= n_final
    
    A_path = fill(params_init.A, T_sim)
    A_path[6:end] .= A_final
    
    res = simulate_solow_swan(params_init, k0, s_path, n_path, A_path, T_sim)
    
    params_fin = SolowSwanParameters(params_init.alpha, params_init.delta, s_final, n_final, A_final)
    ss_fin = compute_solow_steady_state(params_fin)
    
    t_shock = 5
    t_axis = 0:(T_sim - 1)
    
    p1 = plot(t_axis, res["k"], color="#8EAD3A", lw=2.5, label="Capital (k)")
    hline!([ss_init["k"]], color=:gray, ls=:dot, label="SS Inicial")
    hline!([ss_fin["k"]], color=:red, ls=:dash, label="SS Final")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.5, label="")
    title!("Capital por trabajador")
    xlabel!("Tiempo")

    p2 = plot(t_axis, res["y"], color="#004C97", lw=2.5, label="Renta (y)")
    hline!([ss_init["y"]], color=:gray, ls=:dot, label="SS Inicial")
    hline!([ss_fin["y"]], color=:red, ls=:dash, label="SS Final")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.5, label="")
    title!("Renta per cápita")
    xlabel!("Tiempo")

    p3 = plot(t_axis, res["c"], color="#7A3E9F", lw=2.5, label="Consumo (c)")
    hline!([ss_init["c"]], color=:gray, ls=:dot, label="SS Inicial")
    hline!([ss_fin["c"]], color=:red, ls=:dash, label="SS Final")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.5, label="")
    title!("Consumo per cápita")
    xlabel!("Tiempo")

    p4 = plot(t_axis, res["gy"], color="#D95319", lw=2.5, label="Crecimiento (gy)")
    vline!([t_shock], color=:grey, ls=:dot, alpha=0.5, label="")
    title!("Tasa de Crecimiento del PIB p.c. (%)")
    xlabel!("Tiempo")
    ylabel!("% de crecimiento")
    
    plot(p1, p2, p3, p4, layout=(2,2), size=(900, 600), 
         plot_title="Ajuste hacia el Nuevo Estado Estacionario", top_margin=10mm)
end


## 3. La Regla de Oro (Golden Rule) de Acumulación de Capital

Dado que un mayor ahorro implica mayor capital y producción de largo plazo, ¿debe una economía ahorrar tanto como sea posible? La respuesta es **no**. 

El objetivo último de la actividad económica es el **consumo** y el bienestar de los hogares, no la acumulación de capital por sí misma. Si ahorramos una fracción muy alta (cercana al $100\%$), la producción será inmensa, pero casi toda se reinvertirá para compensar la depreciación del gigantesco stock de capital, dejando un consumo cercano a cero. Si ahorramos una fracción muy baja, el capital se agotará y la producción será insignificante, resultando también en un consumo residual.

### 3.1 Derivación Analítica
La tasa de ahorro de la **Regla de Oro** ($s^{gold}$) es aquella que maximiza el consumo de estado estacionario:
$$\max_{s} \bar{c}(s) = \bar{y} - (\delta + n)\bar{k}$$
Sustituyendo $\bar{y} = A \bar{k}^\alpha$:
$$\max_{k} \bar{c} = A \bar{k}^\alpha - (\delta + n)\bar{k}$$
La condición de primer orden con respecto a $\bar{k}$ es:
$$\frac{d\bar{c}}{d\bar{k}} = \alpha A \bar{k}^{\alpha-1} - (\delta + n) = 0 \implies \alpha \frac{\bar{y}}{\bar{k}} = \delta + n$$

Multiplicando ambos lados por $\bar{k}/\bar{y}$ y sabiendo que en estado estacionario $s \frac{\bar{y}}{\bar{k}} = \delta + n$, llegamos a:
$$s^{gold} = \alpha$$

*   **Ineficiencia Dinámica:** Si $s > \alpha$, la economía está sobre-acumulando capital. Podría aumentar el consumo tanto hoy como a largo plazo simplemente reduciendo su tasa de ahorro.
*   **Bajo-acumulación:** Si $s < \alpha$, la economía está consumiendo demasiado a corto plazo, de modo que un incremento del ahorro aumentaría el consumo en el futuro.


In [ ]:
# Demostración Visual de la Regla de Oro: Consumo de Estado Estacionario vs Tasa de Ahorro
@manipulate for s_current in 0.05:0.01:0.60

    params = default_calibration(SolowSwanParameters)
    alpha_val = params.alpha

    # 1. Generar una malla de tasas de ahorro entre 0.01 y 0.95
    s_grid = range(0.01, 0.95, length=100)
    c_ss_grid = [compute_solow_steady_state(params, s_val)["c"] for s_val in s_grid]

    # Consumo actual y de la Regla de Oro (s_gold = alpha)
    c_current = compute_solow_steady_state(params, s_current)["c"]
    c_gold = compute_solow_steady_state(params, alpha_val)["c"]

    # 2. Gráfica
    p1 = plot(s_grid, c_ss_grid, color="#7A3E9F", lw=3, label="Consumo de Estado Estacionario (c̄)")

    # Regiones de (in)eficiencia dinámica
    plot!([0.01, alpha_val], [0.0, 0.0], fillrange=maximum(c_ss_grid) * 1.1, fillalpha=0.5,
          color="#E6F2FF", lw=0, label="Bajo-acumulación (Eficiente)")
    plot!([alpha_val, 0.95], [0.0, 0.0], fillrange=maximum(c_ss_grid) * 1.1, fillalpha=0.5,
          color="#FFE6E6", lw=0, label="Sobre-acumulación (Ineficiente)")

    # Punto actual
    scatter!([s_current], [c_current], color=:red, markersize=6,
             label="Ahorro actual (s=$(round(s_current, digits=2)), c=$(round(c_current, digits=3)))")
    vline!([s_current], color=:red, ls=:dot, alpha=0.5, label="")
    hline!([c_current], color=:red, ls=:dot, alpha=0.5, label="")

    # Regla de Oro
    scatter!([alpha_val], [c_gold], color="#8EAD3A", markersize=10, marker=:star,
             label="Regla de Oro (s_gold=α=$(round(alpha_val, digits=2)), c_gold=$(round(c_gold, digits=3)))")
    vline!([alpha_val], color="#8EAD3A", ls=:dash, alpha=0.7, label="")
    hline!([c_gold], color="#8EAD3A", ls=:dash, alpha=0.7, label="")

    title!("La Regla de Oro: Consumo Estacionario vs. Tasa de Ahorro")
    xlabel!("Tasa de Ahorro (s)")
    ylabel!("Consumo Estacionario (c̄)")
    xlims!(0.01, 0.95)

    plot(p1, size=(750, 500), legend=:bottom, legendfontsize=7)
end


## 4. Cuaderno de Bitácora (Actividades para el Alumno)

Responde a las siguientes cuestiones tras interactuar con las simulaciones del modelo de crecimiento de Solow-Swan:

1.  **Mecanismo de Transición y Sacrificio de Consumo**:
    *   Establece una tasa de ahorro final de $s_1 = 0.25$ (partiendo de $s_0 = 0.20$). Observa la gráfica de consumo per cápita. ¿Qué ocurre exactamente en el período $t=5$ (momento del shock)? ¿Por qué cae el consumo instantáneamente si la producción se mantiene igual?
    *   Explica cómo evoluciona el consumo a partir de $t=6$. ¿Por qué se recupera y acaba superando al nivel inicial? Relaciona esto con el concepto de "acumulación de capital".
2.  **La Regla de Oro y la Ineficiencia Dinámica**:
    *   Utiliza el panel de la Regla de Oro. Establece la tasa de ahorro actual en $s = 0.50$. ¿Qué le ocurre al consumo estacionario comparado con el de la Regla de Oro ($s=0.35$)? ¿Por qué se dice que una economía con $s = 0.50$ es dinámicamente ineficiente? ¿Cómo podría esa economía mejorar el bienestar tanto hoy como en el futuro?
3.  **Transición Demográfica y TFP**:
    *   En la simulación de shocks (Sección 2), devuelve el ahorro al $20\%$ y reduce la tasa de crecimiento demográfico de $n_0 = 0.02$ a $n_1 = 0.00$. Describe la dinámica del capital per cápita y de la producción per cápita. ¿Por qué una población constante ($n=0$) eleva la riqueza por trabajador a largo plazo?
    *   Ahora, aplica un incremento permanente de la TFP del $5\%$ ($A = 1.05$). ¿Qué ocurre con la tasa de crecimiento del PIB per cápita en el año del shock y a largo plazo? Explica por qué, en ausencia de crecimiento continuo de la TFP ($g_A = 0$), el crecimiento de la renta per cápita a largo plazo acaba siendo cero en este modelo.


## 5. Buenas Prácticas Aplicadas en este Laboratorio
1.  **Modularización del Modelo**: Las ecuaciones y simulaciones del modelo de Solow-Swan no están dispersas, sino importadas del módulo de biblioteca [`growth.jl`](file:///c:/Users/AntonioRC/Desktop/PIE/src/macroaicomp/models/growth.jl), aislando la visualización del backend computacional.
2.  **Diseño Paramétrico Limpio**: La calibración y los shocks se definen mediante objetos `SolowSwanParameters` y vectores ordenados de tiempo, facilitando la escalabilidad del simulador.
3.  **Higiene del Repositorio**: El cuaderno está integrado en el flujo de trabajo de Git con `nbstripout` y controles linter en el hook de `pre-commit`, previniendo que los gráficos pesados contaminen el repositorio y asegurando un arranque limpio.


## 6. Benchmark de Rendimiento (Fase III)
Evaluamos la velocidad de simulación usando `BenchmarkTools.jl`.

In [ ]:
# Benchmark simulation
T_bench = 100
s_path = fill(0.20, T_bench); n_path = fill(0.02, T_bench); A_path = fill(1.0, T_bench)
@btime simulate_solow_swan($params_base, $ss["k"], $s_path, $n_path, $A_path, $T_bench)
